<a href="https://colab.research.google.com/github/Quasabianth/hw/blob/main/hw3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа 3. О линейной алгебре в Звёздных Войнах.

## Введение

Инженеры Альянса повстанцев разрабатывают навигационный модуль для управления флотом.
Положение кораблей, переходы между системами координат, калибровка датчиков и анализ устойчивости автопилота описываются с помощью методов линейной алгебры.

В этой лабораторной работе необходимо использовать:

- `numpy` и `numpy.linalg` для численных вычислений;
- `sympy` для символьных вычислений и анализа параметризованных систем.

# Задача 1. Градиентный спуск по космосу. (2 балла)

Наш модуль умеет оценивать вероятность нахождения вражеского флота для каждой точки. Напиши простой градиентный спуск при помощи `sympy`, чтобы найти точку, где вероятность нахождения вражеский флота **наименьшая**, чтобы мы могли отправить туда корабли с гражданскими.

### P.S. напоминание определения.

Градиентный спуск - итеративный алгоритм, который позволяет искать локальный минимум/максимум функции.

Идея: а давайте каждый раз делать микро-шаги в сторону антиградиента:

`steps` раз обнови текущую точку по формуле: $x_{t+1} = x_t - \gamma \nabla F(x_t)$, где $\gamma$ - learning rate (lr).

$x_0$ - стартовая точка, задаётся через параметр `start`. `expr` - выражение, для которого запускается градиентный спуск; `variables` - переменные, по которым идёт пересчёт (их может быть несколько).

In [1]:
import sympy as sp


def gradient_descent(expr, variables, start, lr=0.1, steps=100):
    dims = len(variables)
    values = list(zip(variables, start))
    diffs = [sp.diff(expr, i) for i in variables]
    rez = []
    for i in range(steps):
      for j in range(dims):
        values[j] = (values[j][0], values[j][1] - lr * diffs[j].subs(values))
    for i in values:
      rez.append(i[1])
    return rez

In [3]:
# =========================
# Испытания модуля (тесты)
# =========================

def test_quadratic_1d():
    x = sp.Symbol('x')
    expr = (x - 6)**2
    result = gradient_descent(expr, [x], start=[0], lr=0.1, steps=200)
    # Минимум около на х = 6
    assert abs(result[0] - 6) < 1e-3


def test_quadratic_2d():
    # 2д карта
    x, y = sp.symbols('x y')
    expr = (x - 3)**2 + (y + 5)**2  # Минимум на (3, -5)
    result = gradient_descent(expr, [x, y], start=[0, 0], lr=0.1, steps=200)
    assert abs(result[0] - 3) < 1e-3
    assert abs(result[1] + 5) < 1e-3


def test_flat_minimum():
    x = sp.Symbol('x')
    expr = x**4
    result = gradient_descent(expr, [x], start=[2], lr=0.05, steps=25000)
    assert abs(result[0]) < 1e-2


def test_already_optimal():
    x = sp.Symbol('x')
    expr = (x - 5)**2
    result = gradient_descent(expr, [x], start=[5], lr=0.1, steps=10)
    assert abs(result[0] - 5) < 1e-9

test_quadratic_1d()
test_quadratic_2d()
test_flat_minimum()
test_already_optimal()

## Задача 2. Калибровка навигационного модуля X-wing (2 балла)

Инженеры Альянса повстанцев обнаружили, что навигационный модуль истребителя **X-wing** выполняет линейное преобразование координат сенсоров.  
Это преобразование описывается матрицей

$$
A =
\begin{pmatrix}
a & b \\
c & d
\end{pmatrix}
$$

Чтобы восстановить исходные координаты объектов в пространстве, необходимо вычислить **обратное преобразование**, то есть **обратную матрицу** $A^{-1}$.

Обратная матрица должна удовлетворять условию

$$
A \cdot A^{-1} = I,
$$

где $I$ — единичная матрица.

Для матрицы

$$
\begin{pmatrix}
a & b \\
c & d
\end{pmatrix}
$$

обратная существует **только если её определитель не равен нулю**:

$$
\det(A) = ad - bc \ne 0.
$$

В этом случае обратная матрица вычисляется по формуле

$$
A^{-1} =
\frac{1}{ad-bc}
\begin{pmatrix}
d & -b \\
-c & a
\end{pmatrix}.
$$

### Задание

Напишите функцию на Python, которая:

1. принимает **матрицу $2 \times 2$** в виде списка списков;
2. вычисляет её **обратную матрицу**;
3. возвращает `None`, если матрица **необратима** (то есть её определитель равен нулю).

In [5]:
def inverse_2x2(matrix: list[list[float]]) -> list[list[float]] | None:
    inverse = [[0 for _ in range(2)] for _ in range(2)]
    det = matrix[0][0] * matrix[1][1] - matrix[0][1] * matrix[1][0]
    if det == 0:
      return None
    inverse[0][0] = matrix[1][1] / det
    inverse[0][1] = -matrix[0][1] / det
    inverse[1][0] = -matrix[1][0] / det
    inverse[1][1] = matrix[0][0] / det
    return inverse

## Задача 3. Анализ сигналов навигационного сенсора (2 балла)

Навигационные системы кораблей Альянса получают сигналы от сенсоров, которые можно представить в виде **векторов** или **матриц измерений**.  
Чтобы оценить силу сигнала или величину ошибки измерений, инженеры используют **нормы**.

Норма — это численная характеристика, которая измеряет «размер» или «длину» вектора (или матрицы).

В этой задаче нужно реализовать вычисление нескольких типов норм, часто используемых в линейной алгебре и машинном обучении.

### Типы норм

Пусть задан массив элементов $x_1, x_2, \dots, x_n$.

**L1-норма**

$$
\|x\|_1 = \sum_{i=1}^{n} |x_i|
$$

**L2-норма**

$$
\|x\|_2 = \sqrt{\sum_{i=1}^{n} x_i^2}
$$

**Норма Фробениуса**

Для матрицы $A$:

$$
\|A\|_F = \sqrt{\sum_{i,j} A_{ij}^2}
$$


### Задание

Реализуйте функцию

```python
compute_norm(arr, norm_type)

In [6]:
import numpy as np

def compute_norm(arr: np.ndarray, norm_type: str) -> float:
    x = 0
    if norm_type == 'l1':
      for i in arr:
        x += abs(i)
      return x
    elif norm_type == 'l2':
      for i in arr:
        x = i ** 2
      return np.sqrt(x)
    return

## Задача 4. Декомпозиция матрицы навигационной системы (LU-разложение) (3 балла).

Навигационный компьютер **звёздного разрушителя Империи** использует матричную модель для расчёта траектории флота.  
Матрица коэффициентов системы обозначается

$$
A.
$$

Для ускорения вычислений эта матрица раскладывается в произведение двух более простых матриц:

$$
A = LU,
$$

где

- $L$ — **нижнетреугольная матрица** (элементы выше диагонали равны нулю),
- $U$ — **верхнетреугольная матрица** (элементы ниже диагонали равны нулю).

Такое разложение называется **LU-разложением** и широко используется при решении систем линейных уравнений.

В этой задаче необходимо реализовать LU-разложение по **алгоритму Дулиттла**.

### Алгоритм Дулиттла

В алгоритме Дулиттла:

- диагональные элементы матрицы $L$ равны 1:

$$
L_{ii} = 1
$$

- элементы матрицы $U$ вычисляются по формуле

$$
U_{ij} =
A_{ij} -
\sum_{k=1}^{i-1} L_{ik}U_{kj}
$$

- элементы матрицы $L$ вычисляются как

$$
L_{ij} =
\frac{1}{U_{jj}}
\left(
A_{ij} -
\sum_{k=1}^{j-1} L_{ik}U_{kj}
\right)
\quad \text{для } i > j
$$

### Задание

Реализуйте функцию

```python
lu_decomposition(A)

In [7]:
import numpy as np
import scipy.linalg

def lu_decomposition(A: list) -> tuple:
	n = len(A)
	matrix = np.array(A).reshape(n, n)
	L = np.eye(n)
	U = np.zeros((n, n))
	for i in range(0, n):
		for j in range(0, n):
			if i > j:
				summ = L[i, :j] @ U[:j, j]
				L[i][j] = (matrix[i][j] - summ) / U[j][j]
			else:
				summ = L[i, :j] @ U[:j, j]
				U[i][j] = (matrix[i][j] - summ)
	return (L, U)

In [11]:
A = [[i * 2 + j + 6 for j in range(3)] for i in range(3)]
print(lu_decomposition(A))
print(scipy.linalg.lu(A))

(array([[1.        , 0.        , 0.        ],
       [1.33333333, 1.        , 0.        ],
       [1.66666667, 2.        , 1.        ]]), array([[ 6.00000000e+00,  7.00000000e+00,  8.00000000e+00],
       [ 0.00000000e+00, -3.33333333e-01, -6.66666667e-01],
       [ 0.00000000e+00,  0.00000000e+00,  5.32907052e-15]]))
(array([[0., 1., 0.],
       [0., 0., 1.],
       [1., 0., 0.]]), array([[1. , 0. , 0. ],
       [0.6, 1. , 0. ],
       [0.8, 0.5, 1. ]]), array([[10. , 11. , 12. ],
       [ 0. ,  0.4,  0.8],
       [ 0. ,  0. ,  0. ]]))


In [12]:
B = [[i + 2 * j + 1 for j in range(3)] for i in range(3)]
print(lu_decomposition(B))
print(scipy.linalg.lu(B))

(array([[1., 0., 0.],
       [2., 1., 0.],
       [3., 2., 1.]]), array([[ 1.,  3.,  5.],
       [ 0., -2., -4.],
       [ 0.,  0.,  0.]]))
(array([[0., 1., 0.],
       [0., 0., 1.],
       [1., 0., 0.]]), array([[1.        , 0.        , 0.        ],
       [0.33333333, 1.        , 0.        ],
       [0.66666667, 0.5       , 1.        ]]), array([[3.        , 5.        , 7.        ],
       [0.        , 1.33333333, 2.66666667],
       [0.        , 0.        , 0.        ]]))


In [ ]:
C = [[2, -1, -2], [-4, 6, 3], [-4, -2, 8]]
print(lu_decomposition(C))
print(scipy.linalg.lu(C))

(array([[ 1.,  0.,  0.],
       [-2.,  1.,  0.],
       [-2., -1.,  1.]]), array([[ 2., -1., -2.],
       [ 0.,  4., -1.],
       [ 0.,  0.,  3.]]))
(array([[0., 0., 1.],
       [1., 0., 0.],
       [0., 1., 0.]]), array([[ 1.  ,  0.  ,  0.  ],
       [ 1.  ,  1.  ,  0.  ],
       [-0.5 , -0.25,  1.  ]]), array([[-4.  ,  6.  ,  3.  ],
       [ 0.  , -8.  ,  5.  ],
       [ 0.  ,  0.  ,  0.75]]))


**Как видно из тестов выше, функция scipy.linalg.lu возвращает 3 аргумента, кроме L и U возвращаются также и матрица A с перестановками, также эта функция работает с матрицами произвольного размера, в то время как наша работает только с квадратными.**

## Задача 5. Проверка стабильности навигационного фильтра (разложение Холецкого) (3 балла).

Навигационные системы кораблей **Галактической Империи** используют статистические модели для обработки данных сенсоров.  
Ковариационная матрица ошибок измерений обозначается

$$
A.
$$

Для ускорения вычислений и проверки корректности модели используется **разложение Холецкого**.

### Разложение Холецкого

Разложение Холецкого применяется к **симметричным положительно определённым матрицам**.

Если матрица $A$ обладает этими свойствами, её можно представить в виде

$$
A = L \cdot L^{T},
$$

где

- $L$ — **нижнетреугольная матрица**,
- $L^{T}$ — **транспонированная матрица**.


### Формулы вычисления

Элементы матрицы $L$ вычисляются последовательно.

Для диагональных элементов:

$$
L_{ii} =
\sqrt{
A_{ii} -
\sum_{k=1}^{i-1} L_{ik}^{2}
}
$$

Для элементов ниже диагонали:

$$
L_{ij} =
\frac{
A_{ij} -
\sum_{k=1}^{j-1} L_{ik}L_{jk}
}{
L_{jj}
}
\quad \text{при } i > j
$$

Элементы выше диагонали равны нулю.


### Задание

Реализуйте функцию

```python
cholesky_decomposition(A)
````

которая выполняет **разложение Холецкого** матрицы $A$.

Функция должна:

* принимать на вход **симметричную положительно определённую матрицу**
  (двумерный список или массив `numpy`);
* вычислять **нижнетреугольную матрицу** $L$;
* возвращать $L$ как **двумерный список чисел типа float**.


Использование функций вида

```
numpy.linalg.cholesky
```

**запрещено**.


### Некорректные входные данные

Функция должна вернуть

```
-1
```

если:

* матрица **не квадратная**;
* матрица **пустая** или имеет неверную структуру;
* матрица **не является положительно определённой**.

Если во время вычислений под знаком корня возникает **отрицательное число**, это означает, что матрица не является положительно определённой, и разложение невозможно.


### Интерпретация

Разложение Холецкого используется в навигационных системах для эффективной работы с ковариационными матрицами ошибок сенсоров.
Этот метод широко применяется в:

* решении систем линейных уравнений,
* фильтрах Калмана,
* статистическом моделировании,
* вычислении обратных матриц.



In [17]:
import numpy as np
def is_symmetrix(A):
    for i in range(len(A)):
      for j in range(i):
        if A[i][j] != A[j][i]:
          return False
    return True

def cholesky_decomposition(A):
    matrix = np.array(A)
    matrix_shape = np.shape(matrix)
    dim_x = matrix_shape[1]
    dim_y = matrix_shape[0]
    L = np.zeros((dim_y, dim_x))
    if dim_y != dim_x:
      return -1
    if len(matrix_shape) != 2 or np.all(matrix == 0):
      return -1
    if not is_symmetrix(matrix):
      return -1
    for i in range(dim_y):
      for j in range(dim_y):
        if (i == j):
          sm = np.sum(L[i, :i] ** 2)
          result = A[i][j] - sm
          if result < 0:
            return -1
          L[i][j] = np.sqrt(result)
        if (i > j):
          sm = L[i, :j] @ L[j, :j]
          L[i][j] = (A[i][j] - sm) / L[j][j]
    return (L, L.T)

In [18]:
A = [[i * 3 + j + 1 for j in range(3)] for i in range(3)]
print(cholesky_decomposition(A))
print(np.linalg.cholesky(A))

-1


LinAlgError: Matrix is not positive definite

In [19]:
B = [[1/(i+j+1) for j in range(3)] for i in range(3)]
print(cholesky_decomposition(B))
print(np.linalg.cholesky(B))

(array([[1.        , 0.        , 0.        ],
       [0.5       , 0.28867513, 0.        ],
       [0.33333333, 0.28867513, 0.0745356 ]]), array([[1.        , 0.5       , 0.33333333],
       [0.        , 0.28867513, 0.28867513],
       [0.        , 0.        , 0.0745356 ]]))
[[1.         0.         0.        ]
 [0.5        0.28867513 0.        ]
 [0.33333333 0.28867513 0.0745356 ]]


In [20]:
C = [[1, 7, 9], [2, 2, 8], [1, 1, 1]]
print(cholesky_decomposition(C))
print(np.linalg.cholesky(C))

-1


LinAlgError: Matrix is not positive definite

**Как видно обе функции дают одинаковый результат, единственное отличие в том, что библиотечна функция выдаёт только само L, без L.T, при отрицательно определённой матрице наш код выдаёт код ошибки, а cholesky ложит программу.**

## Задача 6. Анализ сигнала дальнего сканера (SVD-разложение) (3 балла).

Дальний сканер корабля **Millennium Falcon** регистрирует двумерный сигнал от удалённых объектов.  
Полученные данные можно представить в виде матрицы

$$
A =
\begin{pmatrix}
a_{11} & a_{12} \\
a_{21} & a_{22}
\end{pmatrix}.
$$

Для анализа структуры сигнала инженеры используют **сингулярное разложение (SVD)**.  
Оно позволяет выделить основные направления сигнала и его интенсивность.


### Сингулярное разложение

Любая вещественная матрица может быть представлена в виде

$$
A = U \, \Sigma \, V^{T},
$$

где

- $U$ — **ортогональная матрица** левых сингулярных векторов,
- $V$ — **ортогональная матрица** правых сингулярных векторов,
- $\Sigma$ — диагональная матрица сингулярных значений

$$
\Sigma =
\begin{pmatrix}
\sigma_1 & 0 \\
0 & \sigma_2
\end{pmatrix}.
$$

### Метод Якоби

Для приближённого вычисления SVD можно использовать **вращение Якоби**.

Матрица вращения имеет вид

$$
J =
\begin{pmatrix}
\cos\theta & -\sin\theta \\
\sin\theta & \cos\theta
\end{pmatrix}.
$$

Одно вращение позволяет приблизительно диагонализовать матрицу $A^TA$, что даёт оценку сингулярных значений.

В этой задаче требуется выполнить **один шаг вращения Якоби** (без последующих итераций).

### Задание

Реализуйте функцию

```python
svd_2x2(A)
````

которая:

* принимает матрицу `A` размера **2×2** (`numpy.ndarray`);
* вычисляет **приближённое сингулярное разложение**;
* выполняет **один шаг вращения Якоби**.

Использовать можно только **базовые операции NumPy**:

* умножение матриц,
* транспонирование,
* элемент-wise операции.

Использование функций вида

```
numpy.linalg.svd
```

**запрещено**.

### Возвращаемое значение

Функция должна вернуть кортеж

```python
(U, S, Vt)
```

где

* `U` — ортогональная матрица $2 \times 2$ (левые сингулярные векторы),
* `S` — массив длины 2 со **сингулярными значениями**,
* `Vt` — транспонированная матрица правых сингулярных векторов.

Приближённое равенство должно выполняться:

$$
A \approx U \cdot \mathrm{diag}(S) \cdot V^{T}.
$$


### Интерпретация

SVD используется для анализа структуры сигналов и выделения главных направлений данных.
В навигационных системах космических кораблей этот метод помогает:

* отделять полезный сигнал от шума,
* выполнять сжатие данных сенсоров,
* анализировать геометрию наблюдаемых объектов.



In [21]:
import numpy as np

def svd_2x2_singular_values(A: np.ndarray) -> tuple:
    U = np.zeros((2, 2))
    B = A.T @ A
    theta = 0.5 * np.arctan2(2 * B[1][0], B[0][0] - B[1][1])
    V = np.array([[np.cos(theta), -np.sin(theta)],
     [np.sin(theta), np.cos(theta)]])
    Eigen = V.T @ B @ V
    sigma1, sigma2 = 0, 0
    if Eigen[0][0] > 0:
        sigma1 = np.sqrt(Eigen[0][0])
    if Eigen[1][1] > 0:
        sigma2 = np.sqrt(Eigen[1][1])
    Sigmas = np.array([sigma1, sigma2])
    for i in range(2):
      U[:, i] = (A @ V[:, i]) / Sigmas[i] if Sigmas[i] != 0 else 0
    return (U, Sigmas, V.T)


In [22]:
A = np.array([[i * 3 + j + 1 for j in range(2)] for i in range(2)])
print(svd_2x2_singular_values(A))
print(np.linalg.svd(A))

(array([[ 0.32453643,  0.9458732 ],
       [ 0.9458732 , -0.32453643]]), array([6.76782894, 0.44327362]), array([[ 0.60699365,  0.79470668],
       [-0.79470668,  0.60699365]]))
SVDResult(U=array([[-0.32453643, -0.9458732 ],
       [-0.9458732 ,  0.32453643]]), S=array([6.76782894, 0.44327362]), Vh=array([[-0.60699365, -0.79470668],
       [ 0.79470668, -0.60699365]]))


In [23]:
B = np.array([[i + j + 1 for j in range(2)] for i in range(2)])
print(svd_2x2_singular_values(B))
print(np.linalg.svd(B))

(array([[ 0.52573111,  0.85065081],
       [ 0.85065081, -0.52573111]]), array([4.23606798, 0.23606798]), array([[ 0.52573111,  0.85065081],
       [-0.85065081,  0.52573111]]))
SVDResult(U=array([[-0.52573111, -0.85065081],
       [-0.85065081,  0.52573111]]), S=array([4.23606798, 0.23606798]), Vh=array([[-0.52573111, -0.85065081],
       [ 0.85065081, -0.52573111]]))


In [24]:
C = np.array([[2, -1], [-4, 6]])
print(svd_2x2_singular_values(C))
print(np.linalg.svd(C))

(array([[ 0.26542273,  0.96413213],
       [-0.96413213,  0.26542273]]), array([7.47356399, 1.07043975]), array([[ 0.58705244, -0.8095489 ],
       [ 0.8095489 ,  0.58705244]]))
SVDResult(U=array([[-0.26542273,  0.96413213],
       [ 0.96413213,  0.26542273]]), S=array([7.47356399, 1.07043975]), Vh=array([[-0.58705244,  0.8095489 ],
       [ 0.8095489 ,  0.58705244]]))


**Обе функции для случая 2x2 работают корректно, единственно отличие в том, что наши коэффиенты в U и в V это -коэффициенты в разложении с помощью функции svd**

## Задача 7. Перекалибровка сенсорной матрицы (QR-разложение) (3 балла)

Перед гиперпространственным прыжком инженеры **Альянса повстанцев** анализируют данные сенсорной матрицы корабля **X-wing**.  
Измерения записываются в виде матрицы

$$
A.
$$

Чтобы выделить независимые направления измерений и упростить дальнейшие вычисления, матрицу представляют в виде **QR-разложения**.

### QR-разложение

Любая матрица $A$ может быть представлена в виде

$$
A = Q R,
$$

где

- $Q$ — **ортогональная матрица**, столбцы которой образуют **ортонормированный базис**  
  $$
  Q^T Q = I
  $$
- $R$ — **верхнетреугольная матрица**.

### Метод Грама–Шмидта

Чтобы построить матрицу $Q$, используется **процесс Грама–Шмидта**.  
Пусть столбцы матрицы $A$ обозначены

$$
a_1, a_2, \dots, a_n.
$$

Тогда ортогональные векторы вычисляются следующим образом.

Первый столбец:

$$
q_1 = \frac{a_1}{\|a_1\|}
$$

Для каждого следующего столбца:

$$
u_k =
a_k -
\sum_{j=1}^{k-1}
\langle a_k, q_j \rangle q_j
$$

$$
q_k = \frac{u_k}{\|u_k\|}
$$

После построения матрицы $Q$ элементы матрицы $R$ определяются как

$$
R = Q^T A.
$$

### Задание

Реализуйте функцию

```python
qr_decomposition(A)
````

которая:

* принимает матрицу `A`;
* выполняет **QR-разложение с использованием процесса Грама–Шмидта**;
* возвращает кортеж

```python
(Q, R)
```

где

* `Q` — матрица со **столбцами-ортонормированными векторами**;
* `R` — **верхнетреугольная матрица**;
* выполняется равенство

$$
A = Q @ R.
$$

### Ограничения

* реализация должна использовать **алгоритм Грама–Шмидта**;
* запрещается использовать готовые функции типа

```
numpy.linalg.qr
```

### Интерпретация

QR-разложение применяется для:

* решения систем линейных уравнений,
* задач наименьших квадратов,
* ортогонализации признаков,
* устойчивых численных алгоритмов.

В навигационных системах космических кораблей QR-разложение помогает отделять независимые направления сигналов сенсоров и стабилизировать вычисления траектории.


In [26]:
import numpy as np

def qr_decomposition(A):
	matrix = np.array(A)
	shape_of_matrix = np.shape(A)
	dim_x = shape_of_matrix[1]
	dim_y = shape_of_matrix[0]
	Q = np.zeros((dim_y, dim_x))
	U = np.zeros((dim_y, dim_x))
	col = [matrix[:, j] for j in range(dim_x)]
	dist_of_col = [np.linalg.norm(col[j]) for j in range(dim_x)]
	for j in range(dim_x):
		if j == 0:
			Q[:, j] = col[j] / dist_of_col[0]
		else:
			U[:, j] = col[j] - np.sum([col[j] @ Q[:, k] * Q[:, k] for k in range(j)])
			Q[:, j] = U[:, j] / np.linalg.norm(U[:, j])
	QT = Q.T
	R = QT @ matrix
	return (Q, R)

In [27]:
A = [[i * 3 + j + 1 for j in range(3)] for i in range(3)]
print(qr_decomposition(A))
print(np.linalg.qr(A))

(array([[ 0.12309149, -0.74010562, -0.64271292],
       [ 0.49236596, -0.5578408 , -0.5746711 ],
       [ 0.86164044, -0.37557598, -0.50662928]]), array([[ 8.1240384 ,  9.6011363 , 11.07823419],
       [-5.6005007 , -7.2740231 , -8.9475455 ],
       [-6.48780229, -8.21181559, -9.93582889]]))
QRResult(Q=array([[-0.12309149,  0.90453403,  0.40824829],
       [-0.49236596,  0.30151134, -0.81649658],
       [-0.86164044, -0.30151134,  0.40824829]]), R=array([[-8.12403840e+00, -9.60113630e+00, -1.10782342e+01],
       [ 0.00000000e+00,  9.04534034e-01,  1.80906807e+00],
       [ 0.00000000e+00,  0.00000000e+00, -8.88178420e-16]]))


In [28]:
B = [[i + j + 1 for j in range(3)] for i in range(3)]
print(qr_decomposition(B))
print(np.linalg.qr(B))

(array([[ 0.26726124, -0.67378026, -0.60790325],
       [ 0.53452248, -0.57124848, -0.57679114],
       [ 0.80178373, -0.4687167 , -0.54567904]]), array([[ 3.74165739,  5.34522484,  6.94879229],
       [-3.22242731, -4.93617274, -6.64991817],
       [-3.39852266, -5.1288961 , -6.85926953]]))
QRResult(Q=array([[-0.26726124,  0.87287156,  0.40824829],
       [-0.53452248,  0.21821789, -0.81649658],
       [-0.80178373, -0.43643578,  0.40824829]]), R=array([[-3.74165739e+00, -5.34522484e+00, -6.94879229e+00],
       [ 0.00000000e+00,  6.54653671e-01,  1.30930734e+00],
       [ 0.00000000e+00,  0.00000000e+00, -2.22044605e-16]]))


In [29]:
C = [[2, -1, -2], [-4, 6, 3], [-4, -2, 8]]
print(qr_decomposition(C))
print(np.linalg.qr(C))

(array([[ 0.33333333, -0.56568542, -0.84017801],
       [-0.66666667,  0.42426407, -0.51095465],
       [-0.66666667, -0.70710678, -0.18173129]]), array([[ 6.        , -3.        , -8.        ],
       [ 0.        ,  4.5254834 , -3.25269119],
       [ 1.09038776, -1.86208731, -1.30635828]]))
QRResult(Q=array([[-3.33333333e-01,  1.48029737e-16,  9.42809042e-01],
       [ 6.66666667e-01, -7.07106781e-01,  2.35702260e-01],
       [ 6.66666667e-01,  7.07106781e-01,  2.35702260e-01]]), R=array([[-6.        ,  3.        ,  8.        ],
       [ 0.        , -5.65685425,  3.53553391],
       [ 0.        ,  0.        ,  0.70710678]]))


**На примере этой функции заметны отличия в точности подсчёта нашей функции и функции из библиотеки np.linalg, не считая этого обе функции выдают одинаковы ответы с точностью до домножения обоих матриц на минус.**

# Задача 8. Решение систем линейных уравнений (5 баллов).

Все упомянутые выше разложения (QR, SVD, Cholesky, LU) можно применять для решения систем линейных уравнений.

Напиши функцию

```python
solve_linear_system(A: np.array, b: np.array, method)
```

Которая бы находила решение системы Ax=b при помощи заданного разложения.

In [30]:
def solve_linear_system(A: np.array, b: np.array, method="qr"):
    if method == "lu":
      L, U = lu_decomposition(A)
      n = L.shape[0]
      Y = np.zeros(n)
      X = np.zeros(n)
      for i in range(n):
        s = L[i, :i] @ Y[:i]
        Y[i] = (b[i] - s) / L[i, i]
      for i in range(n - 1, -1, -1):
        s = U[i, i+1:] @ X[i+1:]
        X[i] = (Y[i] - s) / U[i, i]
      return X
    if method == "svd":
      U, Sigmas, VT = svd_2x2_singular_values(A)
      V = np.transpose(VT)
      UT = np.transpose(U)
      Sigmas_plus = np.zeros((2, 2))
      Sigmas_plus[0][0] = 1 / Sigmas[0] if Sigmas[0] > 0 else 0
      Sigmas_plus[1][1] = 1 / Sigmas[1] if Sigmas[1] > 0 else 0
      X = V @ Sigmas_plus @ UT @ b
      return X
    if method == "cholesky":
      L, LT = cholesky_decomposition(A)
      n = L.shape[0]
      Y = np.zeros(n)
      X = np.zeros(n)
      for i in range(n):
        s = L[i, :i] @ Y[:i]
        Y[i] = (b[i] - s) / L[i, i]
      for i in range(n - 1, -1, -1):
        s = LT[i, i+1:] @ X[i+1:]
        X[i] = (Y[i] - s) / LT[i, i]
      return X
    if method == "qr":
      Q, R = qr_decomposition(A)
      QT = np.transpose(Q)
      C = QT @ b
      n = Q.shape[0]
      X = np.zeros(n)
      for i in range(n - 1, -1, -1):
        s = R[i, i+1:] @ X[i+1:]
        X[i] = (C[i] - s) / R[i, i]
      return X
    return

In [31]:
import time
A = np.array([[2, 1, -1], [-4, 6, 3], [-4, -2, 8]])
B = np.array([8, 20, 10])
C = np.array([[2, 3], [5, 19]])
D = np.array([9, 10])

Symm = np.array([[2, 1, -1], [1, 6, 3], [-1, 3, 8]])
Big_matrix = np.array([[3 * i + (j ** 2) for i in range(100)] for j in range(100)])
Big_vector = np.array([13 + i ** 1.2 for i in range(100)])
Diag_dom = np.array(([[(i + j + 1) if i != j else (i + j + 1 + 100) for i in range(100)] for j in range(100)]))
t1 = time.time()
print(f'Наши решения: \n{solve_linear_system(A, B, "lu")}\n{solve_linear_system(A, B)}')
t2 = time.time()
print(t2 - t1)
print(f'Решение с помощью numpy: \n{np.linalg.solve(A, B)}')
t3 = time.time()
print(t3 - t2)
print(f'Наши решения: \n{solve_linear_system(Symm, B, "lu")}\n{solve_linear_system(Symm, B)}\n{solve_linear_system(Symm, B, "cholesky")}')
t4 = time.time()
print(t4 - t3)
print(f'Решение с помощью numpy: \n{np.linalg.solve(A, B)}')
t5 = time.time()
print(t5 - t4)
print(solve_linear_system(C, D, "svd"))
print(solve_linear_system(C, D, "lu"))
print(solve_linear_system(Big_matrix, Big_vector, "lu"))
print(solve_linear_system(Big_matrix, Big_vector))
Big_time1 = time.time()
print(solve_linear_system(Diag_dom, Big_vector, "lu"))
Big_time2 = time.time()
print(Big_time2 - Big_time1)
print(solve_linear_system(Diag_dom, Big_vector, "qr"))
Big_time3 = time.time()
print(np.linalg.solve(Diag_dom, Big_vector))
Big_time4 = time.time()
print(Big_time4 - Big_time3)

Наши решения: 
[4.1875     3.95833333 4.33333333]
[18.20647229 10.74007444 13.24069479]
0.0012714862823486328
Решение с помощью numpy: 
[4.1875     3.95833333 4.33333333]
0.0008053779602050781
Наши решения: 
[3.13793103 2.44827586 0.72413793]
[1.66789368 3.21657754 0.01289283]
[3.13793103 2.44827586 0.72413793]
0.0019214153289794922
Решение с помощью numpy: 
[4.1875     3.95833333 4.33333333]
0.0005900859832763672
[ 6.13043478 -1.08695652]
[ 6.13043478 -1.08695652]
[nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan]
[-4.80114687e-03  4.36206836e-05  1.44704560e-05  7.20292103e-06
  4.30721853e-06  2.86379595e-06  2.04098375e-06  1.527749

/tmp/ipykernel_625/4212990499.py:13: RuntimeWarning: divide by zero encountered in scalar divide
  L[i][j] = (matrix[i][j] - summ) / U[j][j]
/tmp/ipykernel_625/4212990499.py:13: RuntimeWarning: invalid value encountered in scalar divide
  L[i][j] = (matrix[i][j] - summ) / U[j][j]


Теперь проведи детальный анализ вариантов решения. Все ли алгоритмы одинаково точны? Какой алгоритм лучше по времени?

Сравни с `numpy.linalg.solve`.

Как мы видим все алгоритмы работают одинаково точно для 9 знаков после запятой, по времени функция из numpy справляется сильно лучше на больших объёмах данных, на маленьких объёмах из-за преобразования получается медленнее.

<b><font color="#FF69B4"> Ваш ответ здесь </font></b>

# Бонусные задания.

## Задача Б1. Расчёт параметров навигационного компьютера (метод Гаусса–Зейделя) (3 балла)

Навигационный компьютер корабля **Millennium Falcon** решает систему линейных уравнений, которая возникает при калибровке сенсоров гипердвигателя.  
Эта система имеет вид

$$
A x = b,
$$

где

- $A$ — матрица коэффициентов системы,
- $x$ — вектор неизвестных параметров,
- $b$ — вектор измерений сенсоров.

Для вычисления решения используется **итерационный метод Гаусса–Зейделя**.

### Метод Гаусса–Зейделя

На каждой итерации компоненты решения обновляются последовательно:

$$
x_i^{(k+1)} =
\frac{1}{a_{ii}}
\left(
b_i
-
\sum_{j<i} a_{ij} x_j^{(k+1)}
-
\sum_{j>i} a_{ij} x_j^{(k)}
\right).
$$

Особенность метода заключается в том, что при вычислении очередной компоненты используются **самые свежие значения**, уже обновлённые на текущей итерации.

### Задание

Реализуйте функцию

```python
gauss_seidel(A, b, n, x_ini=None)
````

где

* `A` — квадратная матрица коэффициентов системы,
* `b` — вектор правой части,
* `n` — число итераций метода,
* `x_ini` — начальное приближение для решения (если не задано, используйте нулевой вектор).

Функция должна вернуть **приближённое решение** $x$ после выполнения `n` итераций метода.


### Предположения

Можно считать, что:

* матрица $A$ **диагонально доминирующая**, что обеспечивает сходимость метода;
* все диагональные элементы $a_{ii} \neq 0$;
* система имеет **единственное решение**.


### Интерпретация

Метод Гаусса–Зейделя постепенно уточняет значения вектора $x$, пока приближение не станет достаточно близким к точному решению системы линейных уравнений.

import numpy as np

def gauss_seidel(A, b, n, x_ini=None):
	return np.zeros_like(b)


In [32]:
# Тесты

## Задача Б2. Навигационный компьютер повстанцев (метод Якоби) (3 балла)

Во время подготовки гиперпространственного прыжка навигационный компьютер базы **Явин IV** должен решить систему линейных уравнений

$$
A x = b,
$$

где

- $A$ — матрица коэффициентов, полученная из навигационных датчиков,
- $x$ — вектор неизвестных параметров траектории,
- $b$ — вектор измеренных сигналов.

Для вычисления решения используется **итерационный метод Якоби**.


### Метод Якоби

На каждой итерации новое значение каждой компоненты вычисляется **только из значений предыдущей итерации**.

Формула обновления имеет вид

$$
x_i^{(k+1)} =
\frac{1}{a_{ii}}
\left(
b_i -
\sum_{j \ne i} a_{ij} x_j^{(k)}
\right),
$$

где

- $a_{ii}$ — диагональный элемент матрицы $A$,
- $a_{ij}$ — остальные элементы строки,
- $x^{(k)}$ — решение на предыдущей итерации.

### Задание

Реализуйте функцию

```python
jacobi(A, b, n)
````

которая:

* принимает

  * `A` — матрицу коэффициентов системы,
  * `b` — вектор правой части,
  * `n` — число итераций метода;

* и возвращает **приближённое решение** $x$ после `n` итераций.

Перед началом итераций необходимо:

* инициализировать вектор решения **нулевым вектором**.

Каждое промежуточное значение решения необходимо **округлять до четырёх знаков после запятой**.

### Интерпретация

Метод Якоби последовательно уточняет параметры навигационной модели, используя значения предыдущей итерации. После нескольких шагов получается приближённое решение системы линейных уравнений.


## Задача Б3. Ортонормированный базис для навигационных векторов (3 балла)

Навигационные датчики корабля **X-wing** фиксируют направления на различные космические объекты.  
Каждое измерение можно представить **двумерным вектором** в пространстве координат сенсоров.

Однако для корректной работы навигационного компьютера необходимо построить **ортонормированный базис** пространства, натянутого на эти векторы. Это позволяет избавиться от линейной зависимости измерений и получить независимые направления.

Для этого используется **процесс Грама–Шмидта**.


### Ортогонализация Грама–Шмидта

Пусть задан набор векторов

$$
v_1, v_2, \dots, v_k.
$$

Алгоритм строит ортогональные векторы следующим образом.

Первый вектор нормализуется:

$$
u_1 = \frac{v_1}{\|v_1\|}.
$$

Для каждого следующего вектора вычисляется ортогональная компонента:

$$
w_i =
v_i -
\sum_{j=1}^{i-1}
\langle v_i, u_j \rangle u_j
$$

После этого вектор нормализуется:

$$
u_i = \frac{w_i}{\|w_i\|}.
$$

Если норма полученного вектора слишком мала, то считается, что вектор **линейно зависим**, и он не включается в базис.

### Задание

Реализуйте функцию

```python
orthonormal_basis(vectors, tol)
````

где

* `vectors` — список **двумерных векторов**;
* `tol` — числовой порог, используемый для проверки **линейной независимости**.

Функция должна:

1. применить **процесс Грама–Шмидта**;
2. построить **ортонормированный базис**;
3. вернуть список векторов, которые

   * имеют **единичную длину**,
   * **ортогональны друг другу**,
   * порождают то же подпространство, что и исходные векторы.



